In [ ]:
'''
    Bond fraction calculation for NCM layered oxide from multicanonical Monte Carlo simulation.

    Created on Sep 19, 2023 at RISM (Shinshu University)
    Last update: Jul 30, 2026 16:42 JST

    Copyright © 2022-2026 Quang Nguyen. All rights reserved.
'''

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker

# Function to calculate thermodynamic variables
def thermo(T, En, BFn, ln_gEBF, N):
    factor    = 96.485332123
    kB        = 8.617333262 * 10 ** (-5)
    Z         = np.zeros(len(T))
    BF        = np.zeros(len(T))
    U         = np.zeros(len(T))
    F         = np.zeros(len(T))
    E2        = np.zeros(len(T))
    S         = np.zeros(len(T))
    Cv        = np.zeros(len(T))
    En_0      = En[0]
    ln_gEBF_0 = ln_gEBF[0, 0]
    gEBF_norm = np.exp(ln_gEBF - ln_gEBF_0)
    for iT in range(len(T)):
        for iE in range(len(En)):
            w = np.exp(- (En[iE] - En_0) / (kB * T[iT]))
            for iBF in range(len(BFn)):
                Z[iT]  += gEBF_norm[iE, iBF] * w
                BF[iT] += BFn[iBF] * gEBF_norm[iE, iBF] * w
                U[iT]  += En[iE] * gEBF_norm[iE, iBF] * w
                E2[iT] += En[iE] ** 2 * gEBF_norm[iE, iBF] * w
        BF[iT] *= 1.0 / Z[iT]
        U[iT]  *= 1.0 / Z[iT]
        E2[iT] *= 1.0 / Z[iT]
        F[iT]   = - kB * T[iT] * np.log(Z[iT]) + En_0
        S[iT]   = (U[iT] - F[iT]) / T[iT]
        Cv[iT]  = (E2[iT] - U[iT] ** 2) / (kB * T[iT] ** 2)
    U  *= (1 / N) * factor
    F  *= (1 / N) * factor
    S  *= (1 / N) * factor
    Cv *= (1 / N) * factor
    return BF, U, F, S, Cv

# Select compound and related data
compound   = 'NCM523'
if compound == 'NCM523':
    cA, cB, cC = 0.5, 0.2, 0.3
elif compound == 'NCM622':
    cA, cB, cC = 0.6, 0.2, 0.2
elif compound == 'NCM811':
    cA, cB, cC = 0.8, 0.1, 0.1
wldata     = '../MCMC_Stage1/'+compound+'_HnDOSvsE.dat'
mucadata1  = '../MCMC_Stage2/'+compound+'_AB_Hist2D.dat'
mucadata2  = '../MCMC_Stage2/'+compound+'_BC_Hist2D.dat'
mucadata3  = '../MCMC_Stage2/'+compound+'_CA_Hist2D.dat'
mucadata4  = '../MCMC_Stage2/'+compound+'_AA_Hist2D.dat'
mucadata5  = '../MCMC_Stage2/'+compound+'_BB_Hist2D.dat'
mucadata6  = '../MCMC_Stage2/'+compound+'_CC_Hist2D.dat'

# Calculate 2-D DOS from normalized 1-D DOS and 2-D histogram
data1      = np.loadtxt(mucadata1, usecols=[1,3,5])
data2      = np.loadtxt(mucadata2, usecols=[1,3,5])
data3      = np.loadtxt(mucadata3, usecols=[1,3,5])
data4      = np.loadtxt(mucadata4, usecols=[1,3,5])
data5      = np.loadtxt(mucadata5, usecols=[1,3,5])
data6      = np.loadtxt(mucadata6, usecols=[1,3,5])
En         = data1[:,0]
BFn        = data1[:,1]
Enx        = data4[:,0]
BFnx       = data4[:,1]
Hist1      = data1[:,2]
Hist2      = data2[:,2]
Hist3      = data3[:,2]
Hist4      = data4[:,2]
Hist5      = data5[:,2]
Hist6      = data6[:,2]
NE, NBF    = 60, 60
if compound == 'NCM622' or compound == 'NCM811':
    NEx, NBFx  = 60, 120
elif compound == 'NCM523':
    NEx, NBFx  = 60, 60
En         = En.reshape((NE, NBF))[:,0]
BFn        = BFn.reshape((NE, NBF))[0,:]
Enx        = Enx.reshape((NEx, NBFx))[:,0]
BFnx       = BFnx.reshape((NEx, NBFx))[0,:]
Hist1      = Hist1.reshape((NE, NBF))
Hist2      = Hist2.reshape((NE, NBF))
Hist3      = Hist3.reshape((NE, NBF))
Hist4      = Hist4.reshape((NEx, NBFx))
Hist5      = Hist5.reshape((NE, NBF))
Hist6      = Hist6.reshape((NE, NBF))
ln_gE      = np.loadtxt(wldata, usecols=[5])
ln_gEBF1   = np.zeros(np.shape(Hist1))
ln_gEBF2   = np.zeros(np.shape(Hist2))
ln_gEBF3   = np.zeros(np.shape(Hist3))
ln_gEBF4   = np.zeros(np.shape(Hist4))
ln_gEBF5   = np.zeros(np.shape(Hist5))
ln_gEBF6   = np.zeros(np.shape(Hist6))
for iE in range(NE):
    for iBF in range(NBF):
        if Hist1[iE, iBF] == 0.0:
            ln_gEBF1[iE, iBF] = 0
        else:
            ln_gEBF1[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist1[iE, iBF])
        if Hist2[iE, iBF] == 0.0:
            ln_gEBF2[iE, iBF] = 0
        else:
            ln_gEBF2[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist2[iE, iBF])
        if Hist3[iE, iBF] == 0.0:
            ln_gEBF3[iE, iBF] = 0
        else:
            ln_gEBF3[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist3[iE, iBF])
        if Hist5[iE, iBF] == 0.0:
            ln_gEBF5[iE, iBF] = 0
        else:
            ln_gEBF5[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist5[iE, iBF])
        if Hist6[iE, iBF] == 0.0:
            ln_gEBF6[iE, iBF] = 0
        else:
            ln_gEBF6[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist6[iE, iBF])
for iE in range(NEx):
    for iBF in range(NBFx):
        if Hist4[iE, iBF] == 0.0:
            ln_gEBF4[iE, iBF] = 0
        else:
            ln_gEBF4[iE, iBF] = (ln_gE[iE] - np.min(ln_gE)) + np.log(Hist4[iE, iBF])
            
# Calculate bond fraction and thermodynamic properties
Tmin, Tmax = 5, 1000
T          = np.linspace(Tmin, Tmax, 200)
N          = 60
BF1, U1, F1, S1, Cv1 = thermo(T, En, BFn, ln_gEBF1, N)
BF2, U2, F2, S2, Cv2 = thermo(T, En, BFn, ln_gEBF2, N)
BF3, U3, F3, S3, Cv3 = thermo(T, En, BFn, ln_gEBF3, N)
BF4, U4, F4, S4, Cv4 = thermo(T, Enx, BFnx, ln_gEBF4, N)
BF5, U5, F5, S5, Cv5 = thermo(T, En, BFn, ln_gEBF5, N)
BF6, U6, F6, S6, Cv6 = thermo(T, En, BFn, ln_gEBF6, N)

# Common settings for all figures
color1, color2, color3 = '#1f77b4', '#ff7f0e', '#2ca02c'
color4, color5, color6 = '#d62728', '#9467bd', '#8c564b'
label1, label2, label3 = 'Ni-Co', 'Co-Mn', 'Mn-Ni'
label4, label5, label6 = 'Ni-Ni', 'Co-Co', 'Mn-Mn'

# Plot AA bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF4, color=color4, linestyle='solid', linewidth=3, label=label4) # AA
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.12 - 0.015/4, 0.24 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.27 - 0.015/4, 0.39 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.54 - 0.015/4, 0.66 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Ni-Ni}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Plot BB bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF5, color=color5, linestyle='solid', linewidth=3, label=label5) # BB
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Co-Co}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Plot CC bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF6, color=color6, linestyle='solid', linewidth=3, label=label6) # CC
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Mn-Mn}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Plot AB bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF1, color=color1, linestyle='solid', linewidth=3, label=label1) # AB
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.12 - 0.015/4, 0.24 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.15 - 0.015/4, 0.27 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.12 - 0.015/4, 0.24 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Ni-Co}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Plot BC bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF2, color=color2, linestyle='solid', linewidth=3, label=label2) # BC
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.03 - 0.015/4, 0.15 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.00 - 0.015/4, 0.12 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Co-Mn}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# Plot CA bond fraction as a function of temperature
plt.figure(figsize=(6.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.2))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(0.03))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.015))
plt.plot(T / 1000, BF3, color=color3, linestyle='solid', linewidth=3, label=label3) # CA
plt.xlim([0.0 - 1/32, 1.0 + 1/32])
if compound == 'NCM523':
    plt.ylim([0.39 - 0.015/4, 0.51 + 0.015/4])
elif compound == 'NCM622':
    plt.ylim([0.30 - 0.015/4, 0.42 + 0.015/4])
elif compound == 'NCM811':
    plt.ylim([0.12 - 0.015/4, 0.24 + 0.015/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$\sigma_{Mn-Ni}$', fontsize=26)
plt.grid(axis='both', which='major', color='k', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()